# FinnHub Bulk OHLC Data Downloader
Downloads OHLC 1-minute data via the bulk download API for Oct-Dec 2025 and saves it to directory `data/finnhub/`.

In [ ]:
import os
import requests
import tarfile
import io
from dotenv import load_dotenv
import tempfile

# Load environment variables to get FINNHUB_API_KEY
load_dotenv()
api_key = os.getenv('FINNHUB_API_KEY')

if not api_key:
    raise ValueError("FINNHUB_API_KEY not found in .env. Please check the .env file.")

# Set up the download directory relative to the notebooks folder
output_dir = '../data/finnhub'
os.makedirs(output_dir, exist_ok=True)

# Target months for the data (Oct, Nov, Dec 2025)
year = 2025
months = [10, 11, 12]

# Base URL for the FinnHub bulk download endpoint
base_url = "https://finnhub.io/api/v1/bulk-download"
exchange = "us"
data_type = "ohlc 1-minute"

print(f"Fetching Bulk OHLC data for year {year}, months {months}...")

for month in months:
    print(f"\nFetching data for {year}-{month:02d}...")
    params = {
        'exchange': exchange,
        'dataType': data_type,
        'year': year,
        'month': month,
        'token': api_key
    }
    
    # Finnhub bulk download endpoint redirects to the actual .tar file
    response = requests.get(base_url, params=params, stream=True)
    
    if response.status_code == 200:
        # Save the file to a temporary location first since tarfile needs a seekable stream
        # or a file object. 
        try:
            with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
                for chunk in response.iter_content(chunk_size=8192):
                    tmp_file.write(chunk)
                tmp_file_path = tmp_file.name
                
            print(f"  -> Downloaded archive. Extracting to {output_dir}...")
            
            # Extract the tar file (could be .tar or .tar.gz)
            with tarfile.open(tmp_file_path, 'r:*') as tar:
                tar.extractall(path=output_dir)
                
            print(f"  -> Successfully extracted data for {year}-{month:02d}.")
            
            # Clean up the temporary file
            os.remove(tmp_file_path)
            
        except tarfile.TarError as e:
             print(f"  -> Error: Downloaded file for {year}-{month:02d} is not a valid tar archive: {e}")
        except Exception as e:
             print(f"  -> An unexpected error occurred: {e}")
    else:
        print(f"  -> Error fetching data for {year}-{month:02d}: Status Code {response.status_code}")
        print(f"     Response message: {response.text}")

print("\nDone!")
